#  Fake Pairing — Text Patients × Image Probabilities
### Paired Dataset for Late Fusion Evaluation

## 1. Imports & Paths

In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ══════════════════════════════════════════════════════════
#  KAGGLE PATHS
# ══════════════════════════════════════════════════════════
BASE     = '/kaggle/input/datasets/habibaamramr/allfiles'
BASE_CSV = '/kaggle/input/datasets/habibaamramr/cleaned-medical-data'

# ── Text side (DistilBERT) ──
BERT_MODEL_PATH      = f'{BASE}/best_bert_model.pt'
DISTILBERT_PROBS     = f'{BASE}/distilbert_probs.npy'
DISEASE_ENCODER_PATH = f'{BASE}/disease_encoder.pkl'
GENDER_ENCODER_PATH  = f'{BASE}/gender_encoder.pkl'

# ── Image side (EfficientNetB3) ──
EFFNET_MODEL_PATH = f'{BASE}/EfficientNetB3_MultiClass_BEST_FINE_TUNED.keras'
EFFNET_PROBS_CSV  = f'{BASE}/EfficientNetB3_MultiClass_test_probabilities.csv'

# ── Medical data ──
TEXT_CSV = f'{BASE_CSV}/cleaned_medical_data.csv'

# ── DistilBERT hyper-params (نفس اللي اتدرب بيها) ──
BERT_MAX_LEN = 256
NUM_CLASSES  = 4

# ── Fusion weights initial (هيتحدد من Grid Search) ──
W_TEXT  = 0.5
W_IMAGE = 0.5

# ── Image classes order as saved in EfficientNet CSV ──
# normal=0, covid=1, pneumonia=2, tuberculosis=3, lung_cancer=4
EFFNET_CLASS_ORDER = ['normal', 'covid-19', 'pneumonia', 'tuberculosis', 'lung cancer']

# ── Alias للـ variables اللي بتستخدمها باقي الـ notebook ──
BERT_PROBS = DISTILBERT_PROBS
EFFNET_CSV = EFFNET_PROBS_CSV
DIS_ENC    = DISEASE_ENCODER_PATH

print("Imports OK ✓")
print(f"BASE     : {BASE}")
print(f"BASE_CSV : {BASE_CSV}")


Imports OK ✓
BASE     : /kaggle/input/datasets/habibaamramr/allfiles
BASE_CSV : /kaggle/input/datasets/habibaamramr/cleaned-medical-data


## 2. Load All Data

In [2]:
# ── Encoders ──
with open(DIS_ENC, 'rb') as f:
    disease_encoder = pickle.load(f)
CLASSES = list(disease_encoder.classes_)
print("Classes:", CLASSES)
# ['covid-19', 'lung cancer', 'pneumonia', 'tuberculosis']

# ── Text patients (600) ──
text_df = pd.read_csv(TEXT_CSV)
print(f"\nText patients: {text_df.shape}")
print(text_df['disease'].value_counts())

# ── EfficientNet test probs (202 rows, including normal) ──
img_df = pd.read_csv(EFFNET_CSV)
print(f"\nImage samples: {img_df.shape}")
print(img_df['true_label'].value_counts())


Classes: ['covid-19', 'lung cancer', 'pneumonia', 'tuberculosis']

Text patients: (602, 7)
disease
lung cancer     156
covid-19        153
tuberculosis    151
pneumonia       142
Name: count, dtype: int64

Image samples: (202, 11)
true_label
covid           42
lung_cancer     41
pneumonia       41
tuberculosis    39
normal          39
Name: count, dtype: int64


## 3. Image Probabilities


In [3]:
assert 'img_df' in dir(), " Cell 2 (Load All Data)!"

# ── شيل الـ normal class من الصور ──
img_no_normal = img_df[img_df['true_label'] != 'normal'].copy().reset_index(drop=True)
print(f"Images after removing normal: {len(img_no_normal)}")

LABEL_MAP = {
    'covid'        : 'covid-19',
    'pneumonia'    : 'pneumonia',
    'tuberculosis' : 'tuberculosis',
    'lung_cancer'  : 'lung cancer'
}
img_no_normal['disease'] = img_no_normal['true_label'].map(LABEL_MAP)
print("\nImage distribution after mapping:")
print(img_no_normal['disease'].value_counts())

prob_cols_5 = ['prob_normal_%', 'prob_covid_%', 'prob_pneumonia_%',
               'prob_tuberculosis_%', 'prob_lung_cancer_%']
probs_5 = img_no_normal[prob_cols_5].values / 100.0

# شيل normal 
probs_4 = probs_5[:, 1:]   # [covid, pneumonia, tuberculosis, lung_cancer]
probs_4 = probs_4 / probs_4.sum(axis=1, keepdims=True)   # re-normalize

# Reorder → [covid-19, lung cancer, pneumonia, tuberculosis]  (زي disease_encoder)
# current:  covid=0, pneumonia=1, tuberculosis=2, lung_cancer=3
# target:   covid=0, lung_cancer=1, pneumonia=2, tuberculosis=3
REORDER = [0, 3, 1, 2]
probs_4_aligned = probs_4[:, REORDER]

img_no_normal['effnet_probs'] = list(probs_4_aligned)
img_no_normal['true_idx']     = [CLASSES.index(d) for d in img_no_normal['disease']]

print(f"\nFinal image probs shape: {probs_4_aligned.shape}")
print(f"Row sum check: {probs_4_aligned[0].sum():.4f}")
print(f"Class order: {CLASSES}")


Images after removing normal: 163

Image distribution after mapping:
disease
covid-19        42
pneumonia       41
lung cancer     41
tuberculosis    39
Name: count, dtype: int64

Final image probs shape: (163, 4)
Row sum check: 1.0000
Class order: ['covid-19', 'lung cancer', 'pneumonia', 'tuberculosis']


## 4 DistilBERT Probabilities

In [4]:
distilbert_probs = np.load(BERT_PROBS)
print(f"DistilBERT probs shape: {distilbert_probs.shape}")  

text_df_sorted = text_df.copy()
text_df_sorted['label'] = [CLASSES.index(d) for d in text_df_sorted['disease']]
_, text_test = train_test_split(text_df_sorted, test_size=0.2, random_state=42,
                                 stratify=text_df_sorted['label'])
text_test = text_test.reset_index(drop=True)
y_test_text = text_test['label'].values

print(f"y_test_text shape: {y_test_text.shape}")   
assert abs(len(y_test_text) - len(distilbert_probs)) <= 1, \
    f"Mismatch: y_test_text={len(y_test_text)} vs distilbert_probs={len(distilbert_probs)}"
print(f"Distribution: {pd.Series(y_test_text).value_counts().sort_index().to_dict()}")

DistilBERT probs shape: (121, 4)
y_test_text shape: (121,)
Distribution: {0: 31, 1: 31, 2: 29, 3: 30}


## 5. Fake Pairing 

In [5]:
def make_paired_dataset(text_df_test, text_probs, y_text,
                        img_df_clean, img_probs_aligned,
                        classes, seed=42):
   # لكل مريض نختار صورة عشوائية من نفس المرض و تاكد انه من نفس الكلاس 

    
    rng = np.random.default_rng(seed)

    available = {}
    for cls in classes:
        mask = img_df_clean['disease'] == cls
        available[cls] = img_df_clean[mask].index.tolist()
        rng.shuffle(available[cls])   # shuffle عشوائي
    
    min_per_class = min(len(v) for v in available.values())
    print(f"Max usable images per class (bottleneck): {min_per_class}")

    paired_rows  = []
    text_p_list  = []
    image_p_list = []
    y_true_list  = []
    used_per_cls = {cls: 0 for cls in classes}

    text_df_test = text_df_test.reset_index(drop=True)  # ضمان 0-based index
    for pos, (i, row) in enumerate(text_df_test.iterrows()):
        cls = classes[int(y_text[pos]) if isinstance(y_text, np.ndarray) else int(row['label'])]
        
        # لو خلصت الصور لهذا الـ class، تخطى
        if used_per_cls[cls] >= len(available[cls]):
            continue
        
        # اختار الصورة الجاية لهذا الـ class
        img_idx = available[cls][used_per_cls[cls]]
        used_per_cls[cls] += 1

        paired_rows.append({
            'text_idx'   : i,
            'img_idx'    : img_idx,
            'disease'    : cls,
            'true_label' : classes.index(cls),
            # text info
            'symptoms'   : row.get('symptoms', ''),
            'age'        : row.get('age', 0),
            'gender'     : row.get('gender', ''),
            # image info
            'effnet_true': img_df_clean.loc[img_idx, 'true_label'],
            'effnet_pred': img_df_clean.loc[img_idx, 'pred_label'] if 'pred_label' in img_df_clean.columns else 'N/A',
        })

        text_p_list.append(text_probs[pos])
        image_p_list.append(img_probs_aligned[img_idx])
        y_true_list.append(classes.index(cls))

    paired_df = pd.DataFrame(paired_rows)
    return (paired_df,
            np.array(text_p_list),
            np.array(image_p_list),
            np.array(y_true_list))


# ── Run الـ pairing ──
paired_df, text_p, image_p, y_paired = make_paired_dataset(
    text_df_test     = text_test,
    text_probs       = distilbert_probs,
    y_text           = y_test_text,
    img_df_clean     = img_no_normal,
    img_probs_aligned= probs_4_aligned,
    classes          = CLASSES,
    seed             = SEED
)

print(f"\nPaired dataset created!")
print(f"   Total pairs      : {len(paired_df)}")
print(f"   text_p shape     : {text_p.shape}")
print(f"   image_p shape    : {image_p.shape}")
print(f"   y_paired shape   : {y_paired.shape}")
print(f"\nClass distribution:")
print(paired_df['disease'].value_counts())


Max usable images per class (bottleneck): 39

Paired dataset created!
   Total pairs      : 121
   text_p shape     : (121, 4)
   image_p shape    : (121, 4)
   y_paired shape   : (121,)

Class distribution:
disease
lung cancer     31
covid-19        31
tuberculosis    30
pneumonia       29
Name: count, dtype: int64


## 6.  Verify pairing

In [6]:
# بتأكد إن كل مريض بياخد صورة من نفس المرض
# map LABEL_MAP in reverse direction for comparison
REVERSE_MAP = {v: k for k, v in LABEL_MAP.items()}
mismatches = paired_df[
    paired_df['effnet_true'].map(LABEL_MAP) != paired_df['disease']
]
print(f"Label mismatches: {len(mismatches)}")   
print("\nSample pairs:")
display_cols = ['disease', 'symptoms', 'age', 'gender', 'effnet_true']
print(paired_df[display_cols].head(10).to_string())

Label mismatches: 0

Sample pairs:
        disease                                                                                                                                                     symptoms  age   gender   effnet_true
0   lung cancer                                                                                              ['fever', 'dyspnea', 'cough', 'weight loss', 'lymphadenopathy']   66   female   lung_cancer
1      covid-19  ['central hypothyroidism', 'endocrine system complications', 'graves disease', 'hypothyroidism', 'isolated high tt', 'non-thyroidal illness syndrome ntis']   48  unknown         covid
2     pneumonia                                                                                             ['asthma', 'cough', 'dyspnea', 'recurrent pneumonia', 'stridor']   19     male     pneumonia
3  tuberculosis                                                                                               ['fever', 'cough', 'abdominal pain', 'weight loss',

## 7. Meta-Learner — Build Full 600-Patient Paired Dataset + Training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split as tts
import copy

# ════════════════════════════════════════════════════════
#  STEP 1 — Fake Pairing على الـ 600 مريض كاملين
#  ┌ Train patients (481) → uniform prior  [0.25, 0.25, 0.25, 0.25]
#  └ Test  patients (121) → distilbert_probs الحقيقية
# ════════════════════════════════════════════════════════

# ── نعيد نفس الـ split اللي اتدرب عليه DistilBERT ──
df_full = text_df.copy().reset_index(drop=True)
df_full['label'] = [CLASSES.index(x) for x in df_full['disease']]

train_df, test_df_meta = tts(
    df_full, test_size=0.2, random_state=42, stratify=df_full['label'])
train_df      = train_df.reset_index(drop=True)
test_df_meta  = test_df_meta.reset_index(drop=True)

assert len(test_df_meta) == len(distilbert_probs), (
    f"Mismatch: test_df={len(test_df_meta)} vs distilbert_probs={len(distilbert_probs)}")
print(f"Train patients : {len(train_df)}")
print(f"Test  patients : {len(test_df_meta)}  (matched with distilbert_probs)")

# ── صور متاحة لكل class مع cycling ──
rng600 = np.random.default_rng(SEED)
available600 = {}
for cls in CLASSES:
    idxs = img_no_normal[img_no_normal['disease'] == cls].index.tolist()
    rng600.shuffle(idxs)
    available600[cls] = idxs
used600 = {cls: 0 for cls in CLASSES}

X_text_list, X_image_list, Y_list = [], [], []

# ── Train patients → uniform prior ──
for _, row in train_df.iterrows():
    cls = row['disease']
    img_idx = available600[cls][used600[cls] % len(available600[cls])]
    used600[cls] += 1
    X_text_list.append(np.full(4, 0.25, dtype=np.float32))
    X_image_list.append(probs_4_aligned[img_idx])
    Y_list.append(row['label'])

# ── Test patients → distilbert_probs الحقيقية ──
for pos, row in test_df_meta.iterrows():
    cls = row['disease']
    img_idx = available600[cls][used600[cls] % len(available600[cls])]
    used600[cls] += 1
    X_text_list.append(distilbert_probs[pos])
    X_image_list.append(probs_4_aligned[img_idx])
    Y_list.append(row['label'])

X_text_600  = np.array(X_text_list,  dtype=np.float32)   # (602, 4)
X_image_600 = np.array(X_image_list, dtype=np.float32)   # (602, 4)
Y_600       = np.array(Y_list,       dtype=np.int64)      # (602,)

print(f"\nFull paired dataset:")
print(f"  X_text  : {X_text_600.shape}")
print(f"  X_image : {X_image_600.shape}")
print(f"  Y       : {Y_600.shape}")
print(f"  Class dist: {dict(zip(CLASSES, np.bincount(Y_600)))}")

# ════════════════════════════════════════════════════════
#  STEP 2 — Concatenate → input (602, 8)
# ════════════════════════════════════════════════════════
X_600 = np.concatenate([X_text_600, X_image_600], axis=1).astype(np.float32)
print(f"\nMeta-Learner input: {X_600.shape}  →  8 features per patient")

# ════════════════════════════════════════════════════════
#  STEP 3 — Train / Val / Test Split
# ════════════════════════════════════════════════════════
X_tv, X_test_m, y_tv, y_test_m = tts(
    X_600, Y_600, test_size=0.2, random_state=SEED, stratify=Y_600)
X_train_m, X_val_m, y_train_m, y_val_m = tts(
    X_tv, y_tv, test_size=0.15, random_state=SEED, stratify=y_tv)

print(f"Train : {len(X_train_m)}  |  Val : {len(X_val_m)}  |  Test : {len(X_test_m)}")

# ════════════════════════════════════════════════════════
#  STEP 4 — PyTorch Dataset & DataLoaders
# ════════════════════════════════════════════════════════
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(FusionDataset(X_train_m, y_train_m), batch_size=16, shuffle=True)
val_loader   = DataLoader(FusionDataset(X_val_m,   y_val_m),   batch_size=32, shuffle=False)
test_loader  = DataLoader(FusionDataset(X_test_m,  y_test_m),  batch_size=32, shuffle=False)

# ════════════════════════════════════════════════════════
#  STEP 5 — Meta-Learner Architecture
# ════════════════════════════════════════════════════════
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")

class MetaLearner(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 4)
        )
    def forward(self, x): return self.net(x)

meta_model = MetaLearner().to(device)
print(meta_model)
print(f"Parameters: {sum(p.numel() for p in meta_model.parameters()):,}")

optimizer = torch.optim.Adam(meta_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=False)
criterion = nn.CrossEntropyLoss()

# ════════════════════════════════════════════════════════
#  STEP 6 — Training Loop + Early Stopping
# ════════════════════════════════════════════════════════
PATIENCE   = 15
MAX_EPOCHS = 150

best_val_acc = 0.0
best_weights = None
patience_cnt = 0
train_accs, val_accs     = [], []
train_losses, val_losses = [], []

print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8} | Status")
print("-" * 70)

for epoch in range(1, MAX_EPOCHS + 1):

    # ── Train ──
    meta_model.train()
    tl, tc, tt = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out  = meta_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        tl += loss.item() * len(yb)
        tc += (out.argmax(1) == yb).sum().item()
        tt += len(yb)

    # ── Validate ──
    meta_model.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = meta_model(xb)
            vl += criterion(out, yb).item() * len(yb)
            vc += (out.argmax(1) == yb).sum().item()
            vt += len(yb)

    ta, va   = tc/tt, vc/vt
    tls, vls = tl/tt, vl/vt
    train_accs.append(ta);   val_accs.append(va)
    train_losses.append(tls); val_losses.append(vls)
    scheduler.step(vls)

    if va > best_val_acc:
        best_val_acc = va
        best_weights = copy.deepcopy(meta_model.state_dict())
        patience_cnt = 0
        status = "✓ best"
    else:
        patience_cnt += 1
        status = f"({patience_cnt}/{PATIENCE})"

    if epoch % 10 == 0 or status == "✓ best":
        print(f"{epoch:>6} | {tls:>10.4f} | {ta*100:>8.2f}% | {vls:>9.4f} | {va*100:>7.2f}% | {status}")

    if patience_cnt >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

meta_model.load_state_dict(best_weights)
print(f"\nBest Val Accuracy: {best_val_acc*100:.2f}%")

# ════════════════════════════════════════════════════════
#  STEP 7 — Training Curves
# ════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train', color='darkorange', linewidth=2)
axes[0].plot(val_losses,   label='Val',   color='steelblue',  linewidth=2)
axes[0].set_title('Loss — Meta-Learner', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([a*100 for a in train_accs], label='Train', color='darkorange', linewidth=2)
axes[1].plot([a*100 for a in val_accs],   label='Val',   color='steelblue',  linewidth=2)
axes[1].set_title('Accuracy — Meta-Learner', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Meta-Learner Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════
#  STEP 8 — Test Set Evaluation
# ════════════════════════════════════════════════════════
meta_model.eval()
meta_preds_list, meta_true_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        meta_preds_list.extend(meta_model(xb.to(device)).argmax(1).cpu().numpy())
        meta_true_list.extend(yb.numpy())

meta_preds = np.array(meta_preds_list)
meta_true  = np.array(meta_true_list)
meta_acc   = accuracy_score(meta_true, meta_preds)
print(f"Meta-Learner Test Accuracy: {meta_acc*100:.2f}%")

# ── Standalone على نفس الـ test set ──
bert_preds_m   = X_test_m[:, :4].argmax(axis=1)
effnet_preds_m = X_test_m[:, 4:].argmax(axis=1)
bert_acc_m     = accuracy_score(y_test_m, bert_preds_m)
effnet_acc_m   = accuracy_score(y_test_m, effnet_preds_m)

torch.save(meta_model.state_dict(), '/kaggle/working/meta_learner.pt')
print("meta_learner.pt saved ✓")


## 8. Final Evaluation — 3 Models Comparison

In [ ]:
print("=" * 52)
print(f"  DistilBERT   (text only)  : {bert_acc_m*100:6.2f}%")
print(f"  EfficientNet (image only) : {effnet_acc_m*100:6.2f}%")
print(f"  Meta-Learner (fusion)     : {meta_acc*100:6.2f}%  ← best")
print("=" * 52)
print(f"  Improvement over best single model: +{(meta_acc - max(bert_acc_m, effnet_acc_m))*100:.2f}%")


## 9. Classification Reports

In [ ]:
for name, preds in [('DistilBERT',    bert_preds_m),
                    ('EfficientNetB3', effnet_preds_m),
                    ('Meta-Learner',   meta_preds)]:
    print(f"\n{'='*52}")
    print(f"  {name}")
    print('='*52)
    print(classification_report(y_test_m, preds, target_names=CLASSES))


## 10. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
configs = [
    ('DistilBERT',     bert_preds_m,   'Oranges'),
    ('EfficientNetB3', effnet_preds_m, 'Blues'),
    ('Meta-Learner',   meta_preds,     'Greens'),
]
for ax, (name, preds, cmap) in zip(axes, configs):
    cm = confusion_matrix(y_test_m, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    acc = accuracy_score(y_test_m, preds)
    ax.set_title(f'{name}\nAcc: {acc*100:.1f}%', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Confusion Matrices — All 3 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 11. Model Accuracy Comparison

In [ ]:
model_names = ['DistilBERT\n(Text Only)',
               'EfficientNetB3\n(Image Only)',
               'Meta-Learner\n(Fusion)']
accuracies  = [bert_acc_m*100, effnet_acc_m*100, meta_acc*100]
colors      = ['darkorange', 'steelblue', 'seagreen']

plt.figure(figsize=(9, 6))
bars = plt.bar(model_names, accuracies, color=colors, width=0.45,
               edgecolor='white', linewidth=1.2)
plt.ylim(0, 115)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Model Comparison — DistilBERT vs EfficientNetB3 vs Meta-Learner',
          fontsize=12, fontweight='bold')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1.5,
             f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()


## 12. Save

In [ ]:
import json

np.save('/kaggle/working/X_test_meta.npy',  X_test_m)
np.save('/kaggle/working/y_test_meta.npy',  y_test_m)
np.save('/kaggle/working/meta_preds.npy',   meta_preds)

config = {
    'model_type'   : 'meta_learner_dense',
    'architecture' : 'Linear(8→64)→BN→ReLU→Drop(0.3)→Linear(64→32)→BN→ReLU→Drop(0.2)→Linear(32→4)',
    'n_total'      : 602,
    'classes'      : CLASSES,
    'meta_acc'     : round(float(meta_acc),     4),
    'bert_acc'     : round(float(bert_acc_m),   4),
    'effnet_acc'   : round(float(effnet_acc_m), 4),
    'best_val_acc' : round(float(best_val_acc), 4),
    'pairing'      : 'fake_pairing_600_cycling',
}
with open('/kaggle/working/fusion_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ All files saved")
print(json.dumps(config, indent=2))


## 13. Drug Recommendation

In [16]:
import ast
from collections import Counter
import pandas as pd
import numpy as np

# ── Drug DB ─────────────────────────────────────
df_med = pd.read_csv(TEXT_CSV)
DRUG_DB = {}

for disease, grp in df_med.groupby('disease'):
    all_drugs = []
    for d in grp['drugs']:
        try:
            all_drugs.extend(ast.literal_eval(d))
        except:
            pass

    DRUG_DB[disease] = [drug for drug, _ in Counter(all_drugs).most_common(5)]

print("\nDRUG DATABASE (Top 5 per disease)\n")

for dis, drugs in DRUG_DB.items():
    print(f"{dis:<15} : {', '.join(drugs)}")


# ── Build Test Cases ───────────────────────────
TEST_CASES = []

for disease in CLASSES:
    disease_idx = CLASSES.index(disease)
    mask = (y_paired == disease_idx)

    if mask.sum() == 0:
        continue

    fused_tmp = W_BEST_TEXT * text_p[mask] + W_BEST_IMAGE * image_p[mask]
    best_pos = int(np.argmax(fused_tmp[:, disease_idx]))

    real_idx = np.where(mask)[0][best_pos]

    TEST_CASES.append({
        "disease": disease,
        "true_idx": disease_idx,
        "paired_idx": real_idx,
        "text_p": text_p[real_idx],
        "image_p": image_p[real_idx],
        "fused_p": W_BEST_TEXT * text_p[real_idx] + W_BEST_IMAGE * image_p[real_idx],
        "row": paired_df.iloc[real_idx],
    })

print("\nNumber of test cases:", len(TEST_CASES))


# ── Comparison Function ────────────────────────
def print_comparison(tc, classes, drug_db):

    true_disease = tc["disease"]
    true_idx = tc["true_idx"]
    row = tc["row"]

    bert_pred = int(np.argmax(tc["text_p"]))
    effnet_pred = int(np.argmax(tc["image_p"]))
    fused_pred = int(np.argmax(tc["fused_p"]))

    symptoms = str(row.get("symptoms", "N/A"))[:70]
    age = row.get("age", "N/A")
    gender = row.get("gender", "N/A")

    print("\n" + "=" * 80)
    print(f"TRUE DISEASE : {true_disease.upper()}")
    print(f"SYMPTOMS     : {symptoms}")
    print(f"AGE / GENDER : {age} / {gender}")
    print("-" * 80)

    # Header
    print(f"{'CLASS':<18}{'DISTILBERT':>15}{'EFFNET':>15}{'FUSION':>15}")
    print("-" * 80)

    for i, cls in enumerate(classes):
        t = tc["text_p"][i] * 100
        v = tc["image_p"][i] * 100
        f = tc["fused_p"][i] * 100

        marker = "<-- TRUE" if i == true_idx else ""

        print(f"{cls:<18}{t:>14.1f}%{v:>14.1f}%{f:>14.1f}% {marker}")

    print("-" * 80)

    print("MODEL PREDICTIONS:")
    print(f"DistilBERT   : {classes[bert_pred]}")
    print(f"EfficientNet : {classes[effnet_pred]}")
    print(f"Fusion       : {classes[fused_pred]}")

    print("\nDRUG RECOMMENDATION (Fusion):")

    pred_disease = classes[fused_pred]
    drugs = drug_db.get(pred_disease, [])
    correct_drugs = drug_db.get(true_disease, [])

    for d in drugs:
        print(f"- {d}")

    if fused_pred != true_idx:
        print("\nNOTE: Wrong prediction")
        print(f"Correct disease: {true_disease}")
        print("Correct drugs:")
        for d in correct_drugs:
            print(f"- {d}")


# ── RUN TESTS ─────────────────────────────────
print("\nEVALUATION START\n")

scores = {"DistilBERT": 0, "EfficientNet": 0, "Fusion": 0}

for tc in TEST_CASES:
    print_comparison(tc, CLASSES, DRUG_DB)

    if np.argmax(tc["text_p"]) == tc["true_idx"]:
        scores["DistilBERT"] += 1

    if np.argmax(tc["image_p"]) == tc["true_idx"]:
        scores["EfficientNet"] += 1

    if np.argmax(tc["fused_p"]) == tc["true_idx"]:
        scores["Fusion"] += 1


# ── FINAL SCOREBOARD ──────────────────────────
print("\n" + "=" * 50)
print("FINAL SCOREBOARD (4 TEST CASES)")
print("=" * 50)

for model, score in scores.items():
    winner = "WINNER" if score == max(scores.values()) else ""
    print(f"{model:<15}: {score}/4 {winner}")

print("=" * 50)


DRUG DATABASE (Top 5 per disease)

covid-19        : remdesivir, molnupiravir, nirmatrelvir, ritonavir, dexamethasone
lung cancer     : cisplatin, carboplatin, paclitaxel, pembrolizumab, etoposide
pneumonia       : amoxicillin, azithromycin, ceftriaxone, levofloxacin, doxycycline
tuberculosis    : isoniazid, rifampicin, pyrazinamide, ethambutol, streptomycin

Number of test cases: 4

EVALUATION START


TRUE DISEASE : COVID-19
SYMPTOMS     : ['fatigue']
AGE / GENDER : 50 / male
--------------------------------------------------------------------------------
CLASS                  DISTILBERT         EFFNET         FUSION
--------------------------------------------------------------------------------
covid-19                    90.8%          98.2%          94.5% <-- TRUE
lung cancer                  0.4%           0.8%           0.6% 
pneumonia                    8.4%           1.0%           4.7% 
tuberculosis                 0.3%           0.0%           0.2% 
-----------------------